In [1]:
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications.resnet50 import preprocess_input
from ultralytics import YOLO
from boxmot.trackers import BotSort
from pathlib import Path
import warnings

I0000 00:00:1787673746.439510   35051 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1787673746.491871   35051 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1787673748.183327   35051 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
SEGMENTATION_MODEL_PATH = "/mnt/d/Computer-Vision/Projects/project1/bestmodel.keras"
REID_WEIGHTS_PATH = Path('osnet_x1_0_msmt17.pt')

YOLO_MODEL_PATH = "/mnt/d/Computer-Vision/Projects/Project2/runs/detect/train-3/weights/best.pt"

In [3]:
IMG_SIZE = (256, 256)

In [4]:
def preprocess_frame_for_segmentation(frame, img_size):
    img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img_tensor = tf.convert_to_tensor(img_rgb, dtype=tf.float32)
    img_resized = tf.image.resize(img_tensor, img_size)
    img_preprocessed = preprocess_input(img_resized)
    img_for_prediction = tf.expand_dims(img_preprocessed, axis=0)
    return img_for_prediction

In [5]:
segmentation_model = tf.keras.models.load_model(SEGMENTATION_MODEL_PATH, compile=False)

W0000 00:00:1787673754.510957   35051 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
W0000 00:00:1787673754.513142   35051 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0a. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
I0000 00:00:1787673754.648357   35051 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5233 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 5060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 12.0a


In [6]:
yolo_model = YOLO(YOLO_MODEL_PATH)

In [7]:
vehicle_tracker = BotSort(
    reid_weights=REID_WEIGHTS_PATH,
    device='0',
    half=False,
    with_reid=False
)

INFO     BotSort: det_thresh=0.3, max_age=30, max_obs=50, min_hits=3, iou_threshold=0.3, per_class=False,          
         asso_func=iou, reid_model=None, track_high_thresh=0.5, track_low_thresh=0.1, new_track_thresh=0.6,        
         track_buffer=30, match_thresh=0.8, proximity_thresh=0.5, appearance_thresh=0.25, cmc_method=ecc,          
         frame_rate=30, fuse_first_associate=False, with_reid=False

In [8]:
VIDEO_PATH = '/mnt/d/Computer-Vision/Projects/Project2/combined model/1_01.mp4'

cap = cv2.VideoCapture(VIDEO_PATH)
if not cap.isOpened():
    print(f"Error: Could not open video file {VIDEO_PATH}")
    exit()

frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out = cv2.VideoWriter('/mnt/d/Computer-Vision/Projects/Project2/combined model/output.mp4',
cv2.VideoWriter_fourcc(*'mp4v'), cap.get(cv2.CAP_PROP_FPS), (frame_width, frame_height))

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # 1-Lines segmentation
    input_tensor = preprocess_frame_for_segmentation(frame, IMG_SIZE)
    prediction_logits = segmentation_model.predict(input_tensor, verbose=0)
    output_mask_tensor = tf.argmax(prediction_logits, axis=-1)
    output_mask = tf.squeeze(output_mask_tensor, axis=0).numpy().astype(np.uint8)
    output_mask_resized = cv2.resize(output_mask, (frame_width, frame_height), interpolation=cv2.INTER_NEAREST)

    # 2-Detect and track cars
    results = yolo_model(frame, stream=True, verbose=False)
    
    detections_for_tracker = []
    for result in results:
        boxes = result.boxes.xyxy.cpu().numpy()
        scores = result.boxes.conf.cpu().numpy()
        labels = result.boxes.cls.cpu().numpy()
        
        if boxes.size > 0:
            detections = np.hstack((boxes, scores[:, np.newaxis], labels[:, np.newaxis]))
            detections_for_tracker.extend(detections)

    tracks = []
    if detections_for_tracker:
        tracks = vehicle_tracker.update(np.array(detections_for_tracker), frame)

    # Result visualizing
    color_mask = np.zeros_like(frame)
    color_mask[output_mask_resized == 1] = [0, 0, 255] 
    combined_frame = cv2.addWeighted(frame, 0.7, color_mask, 0.3, 0)
    
    if len(tracks) > 0:
        for track in tracks:
            x1, y1, x2, y2, track_id, _, _, _ = track
            x1, y1, x2, y2 = map(int, [x1, y1, x2, y2])
            track_id = int(track_id)
            
            cv2.rectangle(combined_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(combined_frame, f"ID: {track_id}", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    out.write(combined_frame)
    cv2.imshow("Lane Segmentation and Vehicle Tracking", combined_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
out.release()
cv2.destroyAllWindows()

I0000 00:00:1787673761.608149   35199 service.cc:153] XLA service 0x7ac48806c210 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1787673761.608200   35199 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5060 Laptop GPU, Compute Capability 12.0a (Driver: 13.3.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.20.0)
I0000 00:00:1787673761.658800   35199 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1787673762.219523   35199 cuda_dnn.cc:461] Loaded cuDNN version 92000
W0000 00:00:1787673763.524348   35333 hlo_rematerialization.cc:3204] Can't reduce memory use below 6.04GiB (6488143473 bytes) by rematerialization; only reduced to 9.70GiB (10419372080 bytes), down from 9.70GiB (10419372080 bytes) originally
W0000 00:00:1787673773.662108   35199 bfc_allocator.cc:502] Allocator (GPU_0_bfc) ran out of memory trying to allocate 9.70GiB (rounded to 10415178496)requ